# sklearn model with tensorflow keras tuner

In [1]:
import sklearn
print(sklearn.__version__)

1.2.2


In [2]:
import re
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from textstat import flesch_reading_ease, gunning_fog
import numpy as np

# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('stopwords')
# nltk.download('vader_lexicon')

def extract_and_aggregate_features(df, text_column):
    """Extracts and aggregates features at the paragraph, sentence, and word levels."""
    # Processing text into paragraphs, sentences, and words
    df['paragraph_lengths'] = df[text_column].apply(lambda x: [len(p.split()) for p in x.split('\n\n') if p.strip()])
    df['sentence_lengths'] = df[text_column].apply(lambda x: [len(s.split()) for s in nltk.sent_tokenize(x)])
    df['word_lengths'] = df[text_column].apply(lambda x: [len(w) for w in x.split()])

    # Aggregating features
    df['paragraph_count'] = df['paragraph_lengths'].apply(len)
    df['sentence_count'] = df['sentence_lengths'].apply(len)
    df['word_count'] = df['word_lengths'].apply(len)

    df['avg_paragraph_length'] = df['paragraph_lengths'].apply(np.mean)
    df['max_paragraph_length'] = df['paragraph_lengths'].apply(max)
    df['min_paragraph_length'] = df['paragraph_lengths'].apply(min)

    df['avg_sentence_length'] = df['sentence_lengths'].apply(np.mean)
    df['max_sentence_length'] = df['sentence_lengths'].apply(max)
    df['min_sentence_length'] = df['sentence_lengths'].apply(min)

    df['avg_word_length'] = df['word_lengths'].apply(np.mean)
    df['max_word_length'] = df['word_lengths'].apply(max)
    df['min_word_length'] = df['word_lengths'].apply(min)

    # Cleaning up DataFrame to remove list columns
    df.drop(['paragraph_lengths', 'sentence_lengths', 'word_lengths'], axis=1, inplace=True)

    return df

def compute_readability_and_sentiment(df, text_column):
    """Computes readability scores and sentiment analysis."""
    df['flesch_reading_ease'] = df[text_column].apply(flesch_reading_ease)
    df['gunning_fog_index'] = df[text_column].apply(gunning_fog)
    sia = SentimentIntensityAnalyzer()
    df['sentiment_score'] = df[text_column].apply(lambda x: sia.polarity_scores(x)['compound'])

    return df


def add_text_features(df, text_column):
    """Main function to aggregate all text processing and feature extraction steps."""
    tqdm.pandas(desc="Extracting and aggregating text features")
    df = extract_and_aggregate_features(df, text_column)
    df = compute_readability_and_sentiment(df, text_column)


    
    return df    

In [3]:

import wordninja
from nltk.corpus import words

# Load a set of valid English words from NLTK for verification (if needed)
word_list = set(words.words())

def segment_text(text, word_list=word_list):
    """
    Segments concatenated words using wordninja, verifying segmentation with an NLTK words list.

    Args:
    text (str): A string of concatenated words.
    word_list (set): A set containing valid words.

    Returns:
    str: Segmented text, or the original text if segmentation results in less common words.
    """
    segmented_words = wordninja.split(text)
    segmented_text = ' '.join(segmented_words)

    # Optional: verify if the original text is a valid word and if the segmented version introduces less common words
    if text in word_list and not all(word in word_list for word in segmented_words):
        return text
    else:
        return segmented_text



def apply_segmentation(df_chunk, text_col='clean_text'):
    """
    Applies text segmentation to the specified 'text' column of a DataFrame chunk using wordninja.

    Args:
    df_chunk (pd.DataFrame): DataFrame chunk containing a text column with concatenated words.

    Returns:
    pd.DataFrame: DataFrame chunk with a new column 'segmented_text' containing segmented text.
    """
    df_chunk['clean_text'] = df_chunk[text_col].apply(lambda x: segment_text(x))
    return df_chunk

In [4]:
def clean_text(df, col_name = 'full_text'):
    """
    Preprocesses text for both training and testing datasets. 
    Includes loading embeddings, building vocabularies, cleaning text among other things.
    
    :param summaries_train: DataFrame with the training data
    :param summaries_test: DataFrame with the testing data
    :param glove_path: path to the GloVe embedding
    :param paragram_path: path to the Paragram embedding
    :param wiki_news_path: path to the Wiki News embedding
    
    :return: Preprocessed DataFrame and list of out-of-vocab words
    """
    print("Starting text cleaning process. \n")

    
    # Lowercase all texts

    df['lowered'] = df[col_name].apply(lambda x: x.lower())

    
    
    def clean_spacing(text):
        
        # Remove spaces before punctuation
        text = re.sub(r'\s+([,.!?])', r'\1', text)

        # Ensure there is one space after punctuation
        text = re.sub(r'([,.!?])([^\s])', r'\1 \2', text)

        return text
    

    df['clean_text'] = df['lowered'].apply(lambda x: clean_spacing(x))


        
    punct = "/-'?!.,#$%\'()*+-/:;<=>@[\\]^_`{|}~" + '""“”’' + '∞θ÷α•à−β∅³π‘₹´°£€\×™√²—–&'
    
    punct_mapping = {"‘": "'", "₹": "e", "´": "'", "°": "", "€": "e", "™": "tm", "√": " sqrt ", "×": "x", "²": "2", "—": "-", "–": "-", "’": "'", "_": "-",
                     "`": "'", '“': '"', '”': '"', '“': '"', "£": "e", '∞': 'infinity', 'θ': 'theta', '÷': '/', 'α': 'alpha', '•': '.', 'à': 'a', '−': '-', 
                     'β': 'beta', '∅': '', '³': '3', 'π': 'pi', }

    def clean_special_chars(text, punct, mapping):
        for p in mapping:
            text = text.replace(p, mapping[p])
        for p in punct:
            text = text.replace(p, f' {p} ')
        specials = {'\u200b': ' ', '…': ' ... ', '\ufeff': '', 'करना': '', 'है': ''}  
        for s in specials:
            text = text.replace(s, specials[s])
        return text
    

    df['clean_text'] = df['clean_text'].apply(lambda x: clean_special_chars(x, punct, punct_mapping))

    df['clean_text'] = df['lowered'].apply(lambda x: clean_spacing(x))


    cont_map = {
        "ain't": "am not","aren't": "are not","can't": "cannot","can't've": "cannot have","'cause": "because",  "could've": "could have",
        "couldn't": "could not","couldn't've": "could not have","didn't": "did not","doesn't": "does not","don't": "do not","hadn't": "had not",
        "hadn't've": "had not have","hasn't": "has not",
        "haven't": "have not","he'd": "he would","he'd've": "he would have","he'll": "he will","he'll've": "he will have","he's": "he is",
        "how'd": "how did","how'd'y": "how do you","how'll": "how will","how's": "how is","I'd": "I would","I'd've": "I would have","I'll": "I will",
        "I'll've": "I will have","I'm": "I am","I've": "I have",
        "isn't": "is not","it'd": "it had","it'd've": "it would have","it'll": "it will", "it'll've": "it will have","it's": "it is","let's": "let us",
        "ma'am": "madam","mayn't": "may not",
        "might've": "might have","mightn't": "might not","mightn't've": "might not have","must've": "must have","mustn't": "must not",
        "mustn't've": "must not have","needn't": "need not","needn't've": "need not have","o'clock": "of the clock","oughtn't": "ought not",
        "oughtn't've": "ought not have","shan't": "shall not","sha'n't": "shall not",
        "shan't've": "shall not have","she'd": "she would","she'd've": "she would have","she'll": "she will","she'll've": "she will have","she's": "she is",
        "should've": "should have","shouldn't": "should not","shouldn't've": "should not have","so've": "so have","so's": "so is","that'd": "that would",
        "that'd've": "that would have","that's": "that is","there'd": "there had","there'd've": "there would have","there's": "there is",
        "they'd": "they would","they'd've": "they would have","they'll": "they will","they'll've": "they will have","they're": "they are",
        "they've": "they have","to've": "to have","wasn't": "was not","we'd": "we had",
        "we'd've": "we would have","we'll": "we will","we'll've": "we will have","we're": "we are","we've": "we have",
        "weren't": "were not","what'll": "what will","what'll've": "what will have",
        "what're": "what are","what's": "what is","what've": "what have","when's": "when is","when've": "when have",
        "where'd": "where did","where's": "where is","where've": "where have","who'll": "who will","who'll've": "who will have","who's": "who is",
        "who've": "who have","why's": "why is",
        "why've": "why have","will've": "will have","won't": "will not","won't've": "will not have","would've": "would have","wouldn't": "would not",
        "wouldn't've": "would not have","y'all": "you all","y'alls": "you alls","y'all'd": "you all would",
        "y'all'd've": "you all would have","y'all're": "you all are","y'all've": "you all have","you'd": "you had","you'd've": "you would have",
        "you'll": "you you will","you'll've": "you you will have","you're": "you are",  "you've": "you have"}

    c_re = re.compile('(%s)' % '|'.join(cont_map.keys()))

    def expandContractions(text, c_re=c_re):
        def replace(match):
            return cont_map[match.group(0)]
        return c_re.sub(replace, text)


#     p = inflect.engine()

    def removeHTML(text):
        """
        Remove HTML tags from a given text string using regex.
    
        Args:
            text (str): The input text string containing HTML tags.
    
        Returns:
            str: The text string with HTML tags removed.
        """
        html = re.compile(r'<.*?>')
        return html.sub('', text)

    def dataPreprocessing(text):
        """
        Process the input text to perform a series of cleaning and formatting tasks,
        including converting numbers to words, removing specific patterns and whitespace,
        and stripping unwanted characters.
    
        Args:
            text (str): The input text string to preprocess.
    
        Returns:
            str: The cleaned and formatted text.
        """
        text = text.lower()
        text = removeHTML(text)
        # text = re.sub(r'\d+', lambda match: p.number_to_words(match.group()) + " ", text)  # Add spaces around the number words
        text = re.sub(r'\s+', ' ', text)  # Normalize multiple spaces to a single space
        text = re.sub(r"@\w+", '', text)
        text = re.sub(r"'\d+", '', text)
        text = re.sub(r"\d+", '', text)
        text = re.sub(r"http\w+", '', text)
        text = re.sub(r"[-_]+", " ", text)  # Replace hyphens and underscores with space
        text = expandContractions(text)
        text = re.sub(r"\.+", ".", text)
        text = re.sub(r"\,+", ",", text)
        text = re.sub(r"[^\w\s]", "", text)  # Remove all non-alphanumeric and non-space characters
        text = re.sub(r"\s+", " ", text)    # Normalize multiple spaces to a single space

        text = text.strip()

        return text

    
    df['clean_text'] = df['clean_text'].apply(lambda x: dataPreprocessing(x))
    
    # Step 8: Generate combined dense vector for each row of text
    
    # df['combined_dense_vector'] = df['clean_text'].apply(lambda x: get_combined_dense_vector(x, embeddings))

    # # Expand the combined dense vector into separate columns
    
    # combined_df = pd.DataFrame(list(df['combined_dense_vector']), 
    #                            columns=[f"dense_vec_{i}" for i in range(df['combined_dense_vector'][0].size)])

    # # Concatenate this new DataFrame with the original DataFrame
    
    # df = pd.concat([df, combined_df], axis=1)

    # # Step 9: Add TF-IDF features to the corrected text
    
    # df.drop(columns=['combined_dense_vector'], inplace=True)
    
    print('Adding Text Features................. \n')
    
    df = add_text_features(df, 'full_text')

    print('Complete................. \n')
    
    return df

In [5]:
import sklearn
print(sklearn.__version__)

1.2.2


In [6]:
laptop = False

if laptop:
    csv_path = '/home/laptop/github/kaggle/scoring/xlnet_hash.csv'

if not laptop:
    csv_path = '/home/jack/github/kaggle/scoring/data/trainining_full (2).csv'

In [9]:
pd.set_option('display.max_columns', None)

deberta_full = pd.read_csv(csv_path)

# deberta_full.drop(columns=['clean_text', 'score', 'labels'], inplace=True, axis=1)

deberta_full.drop(columns=['full_text', 'score'], inplace=True, axis=1)

deberta_full.head()

,essay_id,deberta_prob_0_full,deberta_prob_1_full,deberta_prob_2_full,deberta_prob_3_full,deberta_prob_4_full,deberta_prob_5_full,xlnet_prob_0,xlnet_prob_1,xlnet_prob_2,xlnet_prob_3,xlnet_prob_4,xlnet_prob_5,use_embedding_0,use_embedding_1,use_embedding_2,use_embedding_3,use_embedding_4,use_embedding_5,use_embedding_6,use_embedding_7,use_embedding_8,use_embedding_9,use_embedding_10,use_embedding_11,use_embedding_12,use_embedding_13,use_embedding_14,use_embedding_15,use_embedding_16,use_embedding_17,use_embedding_18,use_embedding_19,use_embedding_20,use_embedding_21,use_embedding_22,use_embedding_23,use_embedding_24,use_embedding_25,use_embedding_26,use_embedding_27,use_embedding_28,use_embedding_29,use_embedding_30,use_embedding_31,use_embedding_32,use_embedding_33,use_embedding_34,use_embedding_35,use_embedding_36,use_embedding_37,use_embedding_38,use_embedding_39,use_embedding_40,use_embedding_41,use_embedding_42,use_embedding_43,use_embedding_44,use_embedding_45,use_embedding_46,use_embedding_47,use_embedding_48,use_embedding_49,use_embedding_50,use_embedding_51,use_embedding_52,use_embedding_53,use_embedding_54,use_embedding_55,use_embedding_56,use_embedding_57,use_embedding_58,use_embedding_59,use_embedding_60,use_embedding_61,use_embedding_62,use_embedding_63,use_embedding_64,use_embedding_65,use_embedding_66,use_embedding_67,use_embedding_68,use_embedding_69,use_embedding_70,use_embedding_71,use_embedding_72,use_embedding_73,use_embedding_74,use_embedding_75,use_embedding_76,use_embedding_77,use_embedding_78,use_embedding_79,use_embedding_80,use_embedding_81,use_embedding_82,use_embedding_83,use_embedding_84,use_embedding_85,use_embedding_86,use_embedding_87,use_embedding_88,use_embedding_89,use_embedding_90,use_embedding_91,use_embedding_92,use_embedding_93,use_embedding_94,use_embedding_95,use_embedding_96,use_embedding_97,use_embedding_98,use_embedding_99,use_embedding_100,use_embedding_101,use_embedding_102,use_embedding_103,use_embedding_104,use_embedding_105,use_embedding_106,use_embedding_107,use_embedding_108,use_embedding_109,use_embedding_110,use_embedding_111,use_embedding_112,use_embedding_113,use_embedding_114,use_embedding_115,use_embedding_116,use_embedding_117,use_embedding_118,use_embedding_119,use_embedding_120,use_embedding_121,use_embedding_122,use_embedding_123,use_embedding_124,use_embedding_125,use_embedding_126,use_embedding_127,use_embedding_128,use_embedding_129,use_embedding_130,use_embedding_131,use_embedding_132,use_embedding_133,use_embedding_134,use_embedding_135,use_embedding_136,use_embedding_137,use_embedding_138,use_embedding_139,use_embedding_140,use_embedding_141,use_embedding_142,use_embedding_143,use_embedding_144,use_embedding_145,use_embedding_146,use_embedding_147,use_embedding_148,use_embedding_149,use_embedding_150,use_embedding_151,use_embedding_152,use_embedding_153,use_embedding_154,use_embedding_155,use_embedding_156,use_embedding_157,use_embedding_158,use_embedding_159,use_embedding_160,use_embedding_161,use_embedding_162,use_embedding_163,use_embedding_164,use_embedding_165,use_embedding_166,use_embedding_167,use_embedding_168,use_embedding_169,use_embedding_170,use_embedding_171,use_embedding_172,use_embedding_173,use_embedding_174,use_embedding_175,use_embedding_176,use_embedding_177,use_embedding_178,use_embedding_179,use_embedding_180,use_embedding_181,use_embedding_182,use_embedding_183,use_embedding_184,use_embedding_185,use_embedding_186,use_embedding_187,use_embedding_188,use_embedding_189,use_embedding_190,use_embedding_191,use_embedding_192,use_embedding_193,use_embedding_194,use_embedding_195,use_embedding_196,use_embedding_197,use_embedding_198,use_embedding_199,use_embedding_200,use_embedding_201,use_embedding_202,use_embedding_203,use_embedding_204,use_embedding_205,use_embedding_206,use_embedding_207,use_embedding_208,use_embedding_209,use_embedding_210,use_embedding_211,use_embedding_212,use_embedding_213,use_embedding_214,use_embedding_215,use_embedding_

In [10]:
# set file paths for either laptop or desktop

if laptop:

    file_config = {

        'vectorizer': '/home/laptop/github/kaggle/scoring/model_data/sklearn/vectorizer.pkl',
        'lda_model': '/home/laptop/github/kaggle/scoring/model_data/sklearn/lda_model.pkl',
        'tf_vectorizer': '/home/laptop/github/kaggle/scoring/model_data/sklearn/tf_vectorizer.pkl',
        'feature_cols': '/home/laptop/github/kaggle/scoring/model_data/sklearn/feature_cols.txt',
        'scaler': '/home/laptop/github/kaggle/scoring/model_data/sklearn/scaler.pkl',   
        'top_corelations': '/home/laptop/github/kaggle/scoring/model_data/sklearn/top_corelations.txt',
        'sklearn_model': '/home/laptop/github/kaggle/scoring/model_data/sklearn/sklearn_model.joblib'}
    
if not laptop:

    file_config = {
        'vectorizer': '/home/jack/github/kaggle/scoring/model_data/sklearn/vectorizer.pkl',
        'lda_model': '/home/jack/github/kaggle/scoring/model_data/sklearn/lda_model.pkl',
        'tf_vectorizer': '/home/jack/github/kaggle/scoring/model_data/sklearn/tf_vectorizer.pkl',
        'feature_cols': '/home/jack/github/kaggle/scoring/model_data/sklearn/feature_cols.txt',
        'scaler': '/home/jack/github/kaggle/scoring/model_data/sklearn/scaler.pkl',   
        'top_corelations': '/home/jack/github/kaggle/scoring/model_data/sklearn/top_corelations.txt',
        'sklearn_model': '/home/jack/github/kaggle/scoring/model_data/sklearn/sklearn_model.joblib'}
    




In [11]:
if laptop:
    csv_path = '/home/laptop/github/kaggle/scoring/data/train.csv'

if not laptop:
    csv_path = '/home/jack/github/kaggle/scoring/data/train.csv'

In [12]:
train = pd.read_csv(csv_path)

In [13]:
train = clean_text(train)

train.head()

Starting text cleaning process. 

Adding Text Features................. 

Complete................. 



,essay_id,full_text,score,lowered,clean_text,paragraph_count,sentence_count,word_count,avg_paragraph_length,max_paragraph_length,min_paragraph_length,avg_sentence_length,max_sentence_length,min_sentence_length,avg_word_length,max_word_length,min_word_length,flesch_reading_ease,gunning_fog_index,sentiment_score
0,000d118,Many people have car where they live. The thin...,3,many people have car where they live. the thin...,many people have car where they live the thing...,1,13,498,498.000000,498,498,38.307692,127,7,4.369478,25,1,57.98,17.33,0.9937
1,000fe60,I am a scientist at NASA that is discussing th...,3,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...,5,21,332,66.400000,98,37,15.809524,48,2,4.018072,11,1,87.55,7.48,0.7705
2,001ab80,People always wish they had the same technolog...,4,people always wish they had the same technolog...,people always wish they had the same technolog...,4,24,550,137.500000,199,85,22.916667,46,9,4.574545,15,1,65.15,11.49,-0.9731
3,001bdc0,"We all heard about Venus, the planet without a...",4,"we all heard about venus, the planet without a...",we all heard about venus the planet without al...,5,20,451,90.200000,165,25,22.550000,38,5,4.982262,20,1,58.32,11.91,0.9702
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3,"dear, state senator\n\nthis is a letter to arg...",dear state senator this is a letter to argue i...,6,15,373,62.166667,118,2,24.933333,76,2,4.873995,14,1,54.66,12.64,0.9771


In [14]:
print(deberta_full.shape, train.shape)

(17307, 525) (17307, 20)


In [15]:
# Combine train and deberta_full on essay_id

train = train.merge(deberta_full, on='essay_id')

In [16]:
train.head()

,essay_id,full_text,score,lowered,clean_text,paragraph_count,sentence_count,word_count,avg_paragraph_length,max_paragraph_length,min_paragraph_length,avg_sentence_length,max_sentence_length,min_sentence_length,avg_word_length,max_word_length,min_word_length,flesch_reading_ease,gunning_fog_index,sentiment_score,deberta_prob_0_full,deberta_prob_1_full,deberta_prob_2_full,deberta_prob_3_full,deberta_prob_4_full,deberta_prob_5_full,xlnet_prob_0,xlnet_prob_1,xlnet_prob_2,xlnet_prob_3,xlnet_prob_4,xlnet_prob_5,use_embedding_0,use_embedding_1,use_embedding_2,use_embedding_3,use_embedding_4,use_embedding_5,use_embedding_6,use_embedding_7,use_embedding_8,use_embedding_9,use_embedding_10,use_embedding_11,use_embedding_12,use_embedding_13,use_embedding_14,use_embedding_15,use_embedding_16,use_embedding_17,use_embedding_18,use_embedding_19,use_embedding_20,use_embedding_21,use_embedding_22,use_embedding_23,use_embedding_24,use_embedding_25,use_embedding_26,use_embedding_27,use_embedding_28,use_embedding_29,use_embedding_30,use_embedding_31,use_embedding_32,use_embedding_33,use_embedding_34,use_embedding_35,use_embedding_36,use_embedding_37,use_embedding_38,use_embedding_39,use_embedding_40,use_embedding_41,use_embedding_42,use_embedding_43,use_embedding_44,use_embedding_45,use_embedding_46,use_embedding_47,use_embedding_48,use_embedding_49,use_embedding_50,use_embedding_51,use_embedding_52,use_embedding_53,use_embedding_54,use_embedding_55,use_embedding_56,use_embedding_57,use_embedding_58,use_embedding_59,use_embedding_60,use_embedding_61,use_embedding_62,use_embedding_63,use_embedding_64,use_embedding_65,use_embedding_66,use_embedding_67,use_embedding_68,use_embedding_69,use_embedding_70,use_embedding_71,use_embedding_72,use_embedding_73,use_embedding_74,use_embedding_75,use_embedding_76,use_embedding_77,use_embedding_78,use_embedding_79,use_embedding_80,use_embedding_81,use_embedding_82,use_embedding_83,use_embedding_84,use_embedding_85,use_embedding_86,use_embedding_87,use_embedding_88,use_embedding_89,use_embedding_90,use_embedding_91,use_embedding_92,use_embedding_93,use_embedding_94,use_embedding_95,use_embedding_96,use_embedding_97,use_embedding_98,use_embedding_99,use_embedding_100,use_embedding_101,use_embedding_102,use_embedding_103,use_embedding_104,use_embedding_105,use_embedding_106,use_embedding_107,use_embedding_108,use_embedding_109,use_embedding_110,use_embedding_111,use_embedding_112,use_embedding_113,use_embedding_114,use_embedding_115,use_embedding_116,use_embedding_117,use_embedding_118,use_embedding_119,use_embedding_120,use_embedding_121,use_embedding_122,use_embedding_123,use_embedding_124,use_embedding_125,use_embedding_126,use_embedding_127,use_embedding_128,use_embedding_129,use_embedding_130,use_embedding_131,use_embedding_132,use_embedding_133,use_embedding_134,use_embedding_135,use_embedding_136,use_embedding_137,use_embedding_138,use_embedding_139,use_embedding_140,use_embedding_141,use_embedding_142,use_embedding_143,use_embedding_144,use_embedding_145,use_embedding_146,use_embedding_147,use_embedding_148,use_embedding_149,use_embedding_150,use_embedding_151,use_embedding_152,use_embedding_153,use_embedding_154,use_embedding_155,use_embedding_156,use_embedding_157,use_embedding_158,use_embedding_159,use_embedding_160,use_embedding_161,use_embedding_162,use_embedding_163,use_embedding_164,use_embedding_165,use_embedding_166,use_embedding_167,use_embedding_168,use_embedding_169,use_embedding_170,use_embedding_171,use_embedding_172,use_embedding_173,use_embedding_174,use_embedding_175,use_embedding_176,use_embedding_177,use_embedding_178,use_embedding_179,use_embedding_180,use_embedding_181,use_embedding_182,use_embedding_183,use_embedding_184,use_embedding_185,use_embedding_186,use_embedding_187,use_embedding_188,use_embedding_189,use_embedding_190,use_embedding_191,use_embedding_192,use_embedding_193,use_embedding_194,use_embedding_195,use_embedding_196,use_embedding_197,use_embedding_198,use_embedding_199,

In [17]:
print(train.shape)

(17307, 544)


In [18]:
import pandas as pd
import textwrap

# Function to format and print the text
def print_formatted_text(df, column_name, index, width=70):
    """
    Print the text from a specified column and row in a formatted and wrapped manner.

    Args:
        df (pd.DataFrame): The DataFrame containing the text data.
        column_name (str): The name of the column containing the text.
        index (int): The index of the row to print.
        width (int, optional): The maximum width of each line. Defaults to 70.
    """
    if index < 0 or index >= len(df):
        print(f"Index {index} is out of bounds for DataFrame with length {len(df)}.")
        return

    wrapped_text = textwrap.fill(df.at[index, column_name], width=width)
    print(f"Text at index {index}:\n{wrapped_text}\n")



In [19]:
# Usage
print_formatted_text(train, 'full_text', 0 ,  width=50)

Text at index 0:
Many people have car where they live. The thing
they don't know is that when you use a car alot of
thing can happen like you can get in accidet
or the smoke that the car has is bad to breath on
if someone is walk but in VAUBAN,Germany they dont
have that proble because 70 percent of vauban's
families do not own cars,and 57 percent sold a car
to move there. Street parkig ,driveways and home
garages are forbidden on the outskirts of freiburd
that near the French and Swiss borders. You
probaly won't see a car in Vauban's streets
because they are completely "car free" but If some
that lives in VAUBAN that owns a car ownership is
allowed,but there are only two places that you can
park a large garages at the edge of the
development,where a car owner buys a space but it
not cheap to buy one they sell the space for you
car for $40,000 along with a home. The vauban
people completed this in 2006 ,they said that this
an example of a growing trend in Europe,The untile
states and s

In [20]:
print_formatted_text(train, 'clean_text', 0 ,  width=50)

Text at index 0:
many people have car where they live the thing
they do not know is that when you use a car alot
of thing can happen like you can get in accidet or
the smoke that the car has is bad to breath on if
someone is walk but in vauban germany they dont
have that proble because percent of vaubans
families do not own cars and percent sold a car to
move there street parkig driveways and home
garages are forbidden on the outskirts of freiburd
that near the french and swiss borders you probaly
will not see a car in vaubans streets because they
are completely car free but if some that lives in
vauban that owns a car ownership is allowed but
there are only two places that you can park a
large garages at the edge of the development where
a car owner buys a space but it not cheap to buy
one they sell the space for you car for along with
a home the vauban people completed this in they
said that this an example of a growing trend in
europe the untile states and some where else are
suburb

In [21]:
df = train.copy()

In [ ]:
# from sklearn.feature_extraction.text import CountVectorizer
# import pandas as pd
# import dill
# import spacy

# # Initialize a CountVectorizer with n-gram range from 1 to 3 (unigrams to trigrams)
# vectorizer = CountVectorizer(ngram_range=(1, 3), max_features=1000, stop_words='english')

# # Initialize and fit the vectorizer
# X_ngrams = vectorizer.fit_transform(df['full_text'])
# df_ngrams = pd.DataFrame(X_ngrams.toarray(), columns=vectorizer.get_feature_names_out())

# # Save the vectorizer
# with open('/home/laptop/github/kaggle/scoring/model_data/sklearn/vectorizer.pkl', 'wb') as f:
#     dill.dump(vectorizer, f)

# # Load the spaCy model
# nlp = spacy.load("en_core_web_sm")

# # Function to count POS tags
# def pos_counts(text):
#     doc = nlp(text)
#     total_tokens = len(doc)
#     pos_counts = doc.count_by(spacy.attrs.POS)
#     return {f'pos_{doc.vocab.strings[pos_id]}': count / total_tokens for pos_id, count in pos_counts.items() if total_tokens > 0}

# # Function to count Named Entities
# def named_entities(text):
#     doc = nlp(text)
#     total_tokens = len(doc)
#     entities = {}
#     for ent in doc.ents:
#         entities[ent.label_] = entities.get(ent.label_, 0) + 1
#     return {f'ner_{label}': count / total_tokens for label, count in entities.items() if total_tokens > 0}

# # Function to count Dependency Tags
# def dependency_tags(text):
#     doc = nlp(text)
#     total_tokens = len(doc)
#     dep_counts = doc.count_by(spacy.attrs.DEP)
#     return {f'dep_{doc.vocab.strings[dep_id]}': count / total_tokens for dep_id, count in dep_counts.items() if total_tokens > 0}

# # Define all possible columns
# all_pos_cols = [f'pos_{pos}' for pos in nlp.get_pipe("tagger").labels]
# all_ent_cols = [f'ner_{ent}' for ent in nlp.get_pipe("ner").labels]
# all_dep_cols = [f'dep_{dep}' for dep in nlp.get_pipe("parser").labels]

# # Extract POS tags, named entities, and dependency tags
# df_pos = df['full_text'].apply(pos_counts).apply(pd.Series).fillna(0)
# df_entities = df['full_text'].apply(named_entities).apply(pd.Series).fillna(0)
# df_deps = df['full_text'].apply(dependency_tags).apply(pd.Series).fillna(0)

# # Ensure all POS, NER, and Dependency columns are present
# for col_list, df_features in [(all_pos_cols, df_pos), (all_ent_cols, df_entities), (all_dep_cols, df_deps)]:
#     for col in col_list:
#         if col not in df_features.columns:
#             df_features[col] = 0

# # Sort columns for consistency
# df_pos = df_pos[sorted(df_pos.columns)]
# df_entities = df_entities[sorted(df_entities.columns)]
# df_deps = df_deps[sorted(df_deps.columns)]

# # Concatenate all feature DataFrames with the original DataFrame
# df = pd.concat([df.reset_index(drop=True), df_ngrams.reset_index(drop=True), 
#                          df_pos.reset_index(drop=True), df_entities.reset_index(drop=True), 
#                          df_deps.reset_index(drop=True)], axis=1)


In [ ]:
# import pandas as pd
# import dill
# import spacy
# import re
# from collections import Counter
# from sklearn.decomposition import LatentDirichletAllocation
# from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
# from sentence_transformers import SentenceTransformer
# from nltk.corpus import stopwords
# from spellchecker import SpellChecker
# import textstat

# # Load spaCy model
# nlp = spacy.load("en_core_web_sm")
# stop_words = set(stopwords.words('english'))
# sentiment_analyzer = SentimentIntensityAnalyzer()
# spell = SpellChecker()
# bert_model = SentenceTransformer('all-MiniLM-L6-v2')

# text_col = 'clean_text'

# # Define feature extraction functions
# def pos_counts(text):
#     doc = nlp(text)
#     pos_counts = doc.count_by(spacy.attrs.POS)
#     return {f'pos_{doc.vocab.strings[pos_id]}': count for pos_id, count in pos_counts.items()}

# def named_entities(text):
#     doc = nlp(text)
#     entities = {ent.label_: ent for ent in doc.ents}
#     return {f'ner_{label}': count for label, count in entities.items()}

# def dependency_tags(text):
#     doc = nlp(text)
#     dep_counts = doc.count_by(spacy.attrs.DEP)
#     return {f'dep_{doc.vocab.strings[dep_id]}': count for dep_id, count in dep_counts.items()}

# def compute_linguistic_features(text):
#     doc = nlp(text)
#     tenses = [token.tag_ for token in doc if token.pos_ == "VERB"]
#     tense_counts = Counter(tenses)
#     most_common_tense = tense_counts.most_common(1)[0][0] if tense_counts else None
#     verb_tense_consistency = tense_counts[most_common_tense] / len(tenses) if tenses else 0
#     passive_count = sum(1 for token in doc if token.dep_ == "nsubjpass")
#     passive_voice_usage = passive_count / len([token for token in doc if token.pos_ == "VERB"]) if doc else 0
#     modal_verbs = {"can", "could", "will", "would", "shall", "should", "may", "might", "must"}
#     modal_count = sum(1 for token in doc if token.lemma_ in modal_verbs)
#     modal_verb_usage = modal_count / len(doc) if doc else 0
#     lexical_diversity = len(set(doc)) / len(doc) if doc else 0
#     dale_chall_readability = textstat.dale_chall_readability_score(text)
#     automated_readability_index = textstat.automated_readability_index(text)
#     coleman_liau_index = textstat.coleman_liau_index(text)
#     conjunction_count = sum(1 for token in doc if token.pos_ in {"CCONJ", "SCONJ"})
#     clause_count = sum(1 for token in doc if token.dep_ in {"csubj", "advcl", "ccomp", "xcomp"})
#     clause_density = clause_count / max(len([token for token in doc if token.dep_ in {"ROOT", "nsubj"}]), 1)
#     return {
#         "verb_tense_consistency": verb_tense_consistency,
#         "passive_voice_usage": passive_voice_usage,
#         "modal_verb_usage": modal_verb_usage,
#         "lexical_diversity": lexical_diversity,
#         "dale_chall_readability": dale_chall_readability,
#         "automated_readability_index": automated_readability_index,
#         "coleman_liau_index": coleman_liau_index,
#         "conjunction_usage": conjunction_count,
#         "clause_density": clause_density
#     }

# def compute_sentiment_features(text):
#     sentiment_scores = sentiment_analyzer.polarity_scores(text)
#     return {
#         "vader_compound": sentiment_scores["compound"],
#         "vader_positive": sentiment_scores["pos"],
#         "vader_negative": sentiment_scores["neg"],
#         "vader_neutral": sentiment_scores["neu"]
#     }

# def extract_bert_embedding(text):
#     embedding = bert_model.encode([text])[0]
#     return {f'bert_dim_{i}': value for i, value in enumerate(embedding)}

# def extract_topic_features(text, lda_model, vectorizer):
#     text_vector = vectorizer.transform([text])
#     topic_distribution = lda_model.transform(text_vector)[0]
#     return {f'topic_{i}': prob for i, prob in enumerate(topic_distribution)}

# def extract_spelling_grammar_errors(text):
#     misspelled = spell.unknown(text.split())
#     return {
#         "spelling_errors": len(misspelled),
#         "grammar_errors": sum(1 for token in nlp(text) if token.dep_ == "pcomp")
#     }

# def compute_advanced_readability_scores(text):
#     return {
#         "linsear_write_formula": textstat.linsear_write_formula(text),
#         "ari_score": textstat.automated_readability_index(text),
#     }

# def apply_feature_extraction(df, lda_model, vectorizer):
#     df_pos = df[text_col].apply(pos_counts).apply(pd.Series).fillna(0)
#     df_entities = df[text_col].apply(named_entities).apply(pd.Series).fillna(0)
#     df_deps = df[text_col].apply(dependency_tags).apply(pd.Series).fillna(0)
#     df_linguistic = df[text_col].apply(compute_linguistic_features).apply(pd.Series)
#     df_sentiment = df[text_col].apply(compute_sentiment_features).apply(pd.Series)
#     df_bert = df[text_col].apply(extract_bert_embedding).apply(pd.Series)
#     df_topics = df[text_col].apply(lambda x: extract_topic_features(x, lda_model, vectorizer)).apply(pd.Series)
#     df_spelling_grammar = df[text_col].apply(extract_spelling_grammar_errors).apply(pd.Series)
#     df_readability = df[text_col].apply(compute_advanced_readability_scores).apply(pd.Series)
#     return pd.concat([df.reset_index(drop=True), df_pos.reset_index(drop=True), 
#                       df_entities.reset_index(drop=True), df_deps.reset_index(drop=True), 
#                       df_linguistic.reset_index(drop=True), df_sentiment.reset_index(drop=True),
#                       df_bert.reset_index(drop=True), df_topics.reset_index(drop=True),
#                       df_spelling_grammar.reset_index(drop=True),
#                       df_readability.reset_index(drop=True)], axis=1)

# # Train LDA topic model
# tf_vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
# tf = tf_vectorizer.fit_transform(df[text_col])
# lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
# lda_model.fit(tf)

# # Apply feature extraction
# df_train_features = apply_feature_extraction(df, lda_model, tf_vectorizer)

# # Save the LDA model and vectorizers
# with open('/home/jack/github/kaggle/scoring/model_data/sklearn/lda_model.pkl', 'wb') as f:
#     dill.dump(lda_model, f)
# with open('/home/jack/github/kaggle/scoring/model_data/sklearn/vectorizer.pkl', 'wb') as f:
#     dill.dump(vectorizer, f)
# with open('/home/jack/github/kaggle/scoring/model_data/sklearn/tf_vectorizer.pkl', 'wb') as f:
#     dill.dump(tf_vectorizer, f)

# # Initialize and fit the CountVectorizer for n-grams
# X_ngrams_train = vectorizer.fit_transform(df[text_col])
# df_ngrams_train = pd.DataFrame(X_ngrams_train.toarray(), columns=vectorizer.get_feature_names_out())

# # Combine all features
# df_train_combined = pd.concat([df_train_features.reset_index(drop=True), df_ngrams_train.reset_index(drop=True)], axis=1)
# df = df_train_combined.copy()


In [22]:
import pandas as pd
import dill
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer
from nltk.corpus import stopwords
from spellchecker import SpellChecker
import textstat
from collections import Counter

# Load spaCy model
nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words('english'))
sentiment_analyzer = SentimentIntensityAnalyzer()
spell = SpellChecker()
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

text_col = 'full_text'

# Initialize a CountVectorizer with n-gram range from 1 to 3 (unigrams to trigrams)
vectorizer = CountVectorizer(ngram_range=(1, 3), max_features=500, stop_words='english')

# Initialize and fit the vectorizer
X_ngrams_train = vectorizer.fit_transform(df[text_col])
df_ngrams_train = pd.DataFrame(X_ngrams_train.toarray(), columns=vectorizer.get_feature_names_out())

# Save the vectorizer
with open(file_config['vectorizer'], 'wb') as f:
    dill.dump(vectorizer, f)

# Train LDA topic model

# tf_vectorizer = TfidfVectorizer(max_features=200, stop_words='english')
# tf = tf_vectorizer.fit_transform(df[text_col])

# lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
# lda_model.fit(tf)

# Save the LDA model and vectorizers

# with open(file_config['lda_model'], 'wb') as f:
#     dill.dump(lda_model, f)
    
# with open(file_config['tf_vectorizer'], 'wb') as f:
#     dill.dump(tf_vectorizer, f)

# Define feature extraction functions
def pos_counts(text):
    doc = nlp(text)
    total_tokens = len(doc)
    pos_counts = doc.count_by(spacy.attrs.POS)
    pos_features = {f'pos_{doc.vocab.strings[pos_id]}_count': count for pos_id, count in pos_counts.items()}
    pos_ratios = {f'pos_{doc.vocab.strings[pos_id]}_ratio': count / total_tokens for pos_id, count in pos_counts.items() if total_tokens > 0}
    return {**pos_features, **pos_ratios}

def named_entities(text):
    doc = nlp(text)
    total_tokens = len(doc)
    entities = {}
    for ent in doc.ents:
        entities[ent.label_] = entities.get(ent.label_, 0) + 1
    entity_features = {f'ner_{label}_count': count for label, count in entities.items()}
    entity_ratios = {f'ner_{label}_ratio': count / total_tokens for label, count in entities.items() if total_tokens > 0}
    return {**entity_features, **entity_ratios}

def dependency_tags(text):
    doc = nlp(text)
    total_tokens = len(doc)
    dep_counts = doc.count_by(spacy.attrs.DEP)
    dep_features = {f'dep_{doc.vocab.strings[dep_id]}_count': count for dep_id, count in dep_counts.items()}
    dep_ratios = {f'dep_{doc.vocab.strings[dep_id]}_ratio': count / total_tokens for dep_id, count in dep_counts.items() if total_tokens > 0}
    return {**dep_features, **dep_ratios}

def compute_linguistic_features(text):
    doc = nlp(text)
    tenses = [token.tag_ for token in doc if token.pos_ == "VERB"]
    tense_counts = Counter(tenses)
    most_common_tense = tense_counts.most_common(1)[0][0] if tense_counts else None
    verb_tense_consistency = tense_counts[most_common_tense] / len(tenses) if tenses else 0

    verbs = [token for token in doc if token.pos_ == "VERB"]
    passive_count = sum(1 for token in doc if token.dep_ == "nsubjpass")
    passive_voice_usage = passive_count / len(verbs) if verbs else 0

    modal_verbs = {"can", "could", "will", "would", "shall", "should", "may", "might", "must"}
    modal_count = sum(1 for token in doc if token.lemma_ in modal_verbs)
    modal_verb_usage = modal_count / len(doc) if len(doc) > 0 else 0

    lexical_diversity = len(set(doc)) / len(doc) if len(doc) > 0 else 0

    dale_chall_readability = textstat.dale_chall_readability_score(text)
    automated_readability_index = textstat.automated_readability_index(text)
    coleman_liau_index = textstat.coleman_liau_index(text)

    conjunction_count = sum(1 for token in doc if token.pos_ in {"CCONJ", "SCONJ"})
    clause_count = sum(1 for token in doc if token.dep_ in {"csubj", "advcl", "ccomp", "xcomp"})
    clause_density = clause_count / max(len([token for token in doc if token.dep_ in {"ROOT", "nsubj"}]), 1)

    return {
        "verb_tense_consistency": verb_tense_consistency,
        "passive_voice_usage": passive_voice_usage,
        "modal_verb_usage": modal_verb_usage,
        "lexical_diversity": lexical_diversity,
        "dale_chall_readability": dale_chall_readability,
        "automated_readability_index": automated_readability_index,
        "coleman_liau_index": coleman_liau_index,
        "conjunction_usage": conjunction_count,
        "clause_density": clause_density
    }

def compute_sentiment_features(text):
    sentiment_scores = sentiment_analyzer.polarity_scores(text)
    return {
        "vader_compound": sentiment_scores["compound"],
        "vader_positive": sentiment_scores["pos"],
        "vader_negative": sentiment_scores["neg"],
        "vader_neutral": sentiment_scores["neu"]
    }

def extract_bert_embedding(text):
    embedding = bert_model.encode([text])[0]
    return {f'bert_dim_{i}': value for i, value in enumerate(embedding)}

def extract_topic_features(text, lda_model, vectorizer):
    text_vector = vectorizer.transform([text])
    topic_distribution = lda_model.transform(text_vector)[0]
    return {f'topic_{i}': prob for i, prob in enumerate(topic_distribution)}

def extract_spelling_grammar_errors(text):
    misspelled = spell.unknown(text.split())
    return {
        "spelling_errors": len(misspelled),
        "grammar_errors": sum(1 for token in nlp(text) if token.dep_ == "pcomp")
    }

def compute_advanced_readability_scores(text):
    return {
        "linsear_write_formula": textstat.linsear_write_formula(text),
        "ari_score": textstat.automated_readability_index(text),
    }

def apply_feature_extraction(df, lda_model=None, vectorizer=None):
    df_pos = df[text_col].apply(pos_counts).apply(pd.Series).fillna(0)
    df_entities = df[text_col].apply(named_entities).apply(pd.Series).fillna(0)
    df_deps = df[text_col].apply(dependency_tags).apply(pd.Series).fillna(0)
    # df_linguistic = df[text_col].apply(compute_linguistic_features).apply(pd.Series)
    # df_sentiment = df[text_col].apply(compute_sentiment_features).apply(pd.Series)

    # df_bert = df[text_col].apply(extract_bert_embedding).apply(pd.Series)
    # df_topics = df[text_col].apply(lambda x: extract_topic_features(x, lda_model, vectorizer)).apply(pd.Series)

    df_spelling_grammar = df[text_col].apply(extract_spelling_grammar_errors).apply(pd.Series)
    df_readability = df[text_col].apply(compute_advanced_readability_scores).apply(pd.Series)
    
    return pd.concat([df.reset_index(drop=True), df_pos.reset_index(drop=True), 
                      df_entities.reset_index(drop=True), df_deps.reset_index(drop=True), 
                    #   df_linguistic.reset_index(drop=True), df_sentiment.reset_index(drop=True),
                      df_spelling_grammar.reset_index(drop=True),
                      df_readability.reset_index(drop=True)], axis=1)    # df_bert.reset_index(drop=True), df_topics.reset_index(drop=True)
                      

# Apply feature extraction
df_train_features = apply_feature_extraction(df) # , lda_model, tf_vectorizer)

# Combine all features
df_train_combined = pd.concat([df_train_features.reset_index(drop=True), df_ngrams_train.reset_index(drop=True)], axis=1)

df = df_train_combined.copy()

/home/jack/envs/scoring/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
df.head()

,essay_id,full_text,score,lowered,clean_text,paragraph_count,sentence_count,word_count,avg_paragraph_length,max_paragraph_length,min_paragraph_length,avg_sentence_length,max_sentence_length,min_sentence_length,avg_word_length,max_word_length,min_word_length,flesch_reading_ease,gunning_fog_index,sentiment_score,deberta_prob_0_full,deberta_prob_1_full,deberta_prob_2_full,deberta_prob_3_full,deberta_prob_4_full,deberta_prob_5_full,xlnet_prob_0,xlnet_prob_1,xlnet_prob_2,xlnet_prob_3,xlnet_prob_4,xlnet_prob_5,use_embedding_0,use_embedding_1,use_embedding_2,use_embedding_3,use_embedding_4,use_embedding_5,use_embedding_6,use_embedding_7,use_embedding_8,use_embedding_9,use_embedding_10,use_embedding_11,use_embedding_12,use_embedding_13,use_embedding_14,use_embedding_15,use_embedding_16,use_embedding_17,use_embedding_18,use_embedding_19,use_embedding_20,use_embedding_21,use_embedding_22,use_embedding_23,use_embedding_24,use_embedding_25,use_embedding_26,use_embedding_27,use_embedding_28,use_embedding_29,use_embedding_30,use_embedding_31,use_embedding_32,use_embedding_33,use_embedding_34,use_embedding_35,use_embedding_36,use_embedding_37,use_embedding_38,use_embedding_39,use_embedding_40,use_embedding_41,use_embedding_42,use_embedding_43,use_embedding_44,use_embedding_45,use_embedding_46,use_embedding_47,use_embedding_48,use_embedding_49,use_embedding_50,use_embedding_51,use_embedding_52,use_embedding_53,use_embedding_54,use_embedding_55,use_embedding_56,use_embedding_57,use_embedding_58,use_embedding_59,use_embedding_60,use_embedding_61,use_embedding_62,use_embedding_63,use_embedding_64,use_embedding_65,use_embedding_66,use_embedding_67,use_embedding_68,use_embedding_69,use_embedding_70,use_embedding_71,use_embedding_72,use_embedding_73,use_embedding_74,use_embedding_75,use_embedding_76,use_embedding_77,use_embedding_78,use_embedding_79,use_embedding_80,use_embedding_81,use_embedding_82,use_embedding_83,use_embedding_84,use_embedding_85,use_embedding_86,use_embedding_87,use_embedding_88,use_embedding_89,use_embedding_90,use_embedding_91,use_embedding_92,use_embedding_93,use_embedding_94,use_embedding_95,use_embedding_96,use_embedding_97,use_embedding_98,use_embedding_99,use_embedding_100,use_embedding_101,use_embedding_102,use_embedding_103,use_embedding_104,use_embedding_105,use_embedding_106,use_embedding_107,use_embedding_108,use_embedding_109,use_embedding_110,use_embedding_111,use_embedding_112,use_embedding_113,use_embedding_114,use_embedding_115,use_embedding_116,use_embedding_117,use_embedding_118,use_embedding_119,use_embedding_120,use_embedding_121,use_embedding_122,use_embedding_123,use_embedding_124,use_embedding_125,use_embedding_126,use_embedding_127,use_embedding_128,use_embedding_129,use_embedding_130,use_embedding_131,use_embedding_132,use_embedding_133,use_embedding_134,use_embedding_135,use_embedding_136,use_embedding_137,use_embedding_138,use_embedding_139,use_embedding_140,use_embedding_141,use_embedding_142,use_embedding_143,use_embedding_144,use_embedding_145,use_embedding_146,use_embedding_147,use_embedding_148,use_embedding_149,use_embedding_150,use_embedding_151,use_embedding_152,use_embedding_153,use_embedding_154,use_embedding_155,use_embedding_156,use_embedding_157,use_embedding_158,use_embedding_159,use_embedding_160,use_embedding_161,use_embedding_162,use_embedding_163,use_embedding_164,use_embedding_165,use_embedding_166,use_embedding_167,use_embedding_168,use_embedding_169,use_embedding_170,use_embedding_171,use_embedding_172,use_embedding_173,use_embedding_174,use_embedding_175,use_embedding_176,use_embedding_177,use_embedding_178,use_embedding_179,use_embedding_180,use_embedding_181,use_embedding_182,use_embedding_183,use_embedding_184,use_embedding_185,use_embedding_186,use_embedding_187,use_embedding_188,use_embedding_189,use_embedding_190,use_embedding_191,use_embedding_192,use_embedding_193,use_embedding_194,use_embedding_195,use_embedding_196,use_embedding_197,use_embedding_198,use_embedding_199,

In [24]:

# Import necessary libraries
import pandas as pd
from itertools import combinations

# Assuming the DataFrame `df` and `score` are already defined
# and excluding irrelevant columns

# Calculate correlations with the target

cor = df.drop(columns=['essay_id', 'full_text', 'clean_text', 'lowered' , 'score'], axis=1) # ,'processed_docs', 'topics',

correlations = cor.corrwith(df['score']).sort_values(ascending=False)



# Set a correlation threshold or select top N features
correlation_threshold = 0.3

top_features = correlations[abs(correlations) > correlation_threshold].index.tolist()

# Print the most correlated features
print("Top correlations with the score:")
print(top_features)

# Generate interaction terms between pairs of top features
for feature1, feature2 in combinations(top_features, 2):
    df[f'interaction_{feature1}_{feature2}'] = df[feature1] * df[feature2]


# Optional: print the list of created interaction terms
interaction_terms = [f'interaction_{feature1}_{feature2}' for feature1, feature2 in combinations(top_features, 2)]
print("Created interaction terms:")
print(interaction_terms)


# export top_features to text file

with open('/home/jack/github/kaggle/scoring/model_data/sklearn/top_corelations.txt', 'w') as f:
    for item in top_features:
        f.write("%s\n" % item)


Top correlations with the score:
['word_count', 'pos_DET_count', 'pos_NOUN_count', 'dep_det_count', 'pos_AUX_count', 'pos_ADJ_count', 'pos_VERB_count', 'pos_PUNCT_count', 'deberta_prob_4_full', 'dep_punct_count', 'deberta_prob_3_full', 'pos_ADP_count', 'dep_prep_count', 'dep_ROOT_count', 'sentence_count', 'dep_pobj_count', 'dep_nsubj_count', 'dep_aux_count', 'dep_advmod_count', 'dep_amod_count', 'spelling_errors', 'pos_ADV_count', 'dep_dobj_count', 'pos_PART_count', 'use_embedding_233', 'use_embedding_356', 'pos_SCONJ_count', 'use_embedding_503', 'dep_advcl_count', 'dep_acomp_count', 'dep_mark_count', 'dep_cc_count', 'pos_CCONJ_count', 'dep_conj_count', 'deberta_prob_5_full', 'dep_relcl_count', 'dep_pcomp_count', 'grammar_errors', 'dep_auxpass_count', 'dep_acl_count', 'dep_nsubjpass_count', 'use_embedding_483', 'pos_PRON_count', 'dep_xcomp_count', 'dep_compound_count', 'dep_neg_count', 'dep_ccomp_count', 'dep_attr_count', 'dep_poss_count', 'use_embedding_141', 'use_embedding_427', 'use

/tmp/ipykernel_104197/3155286109.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'interaction_{feature1}_{feature2}'] = df[feature1] * df[feature2]
/tmp/ipykernel_104197/3155286109.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'interaction_{feature1}_{feature2}'] = df[feature1] * df[feature2]
/tmp/ipykernel_104197/3155286109.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

Created interaction terms:
['interaction_word_count_pos_DET_count', 'interaction_word_count_pos_NOUN_count', 'interaction_word_count_dep_det_count', 'interaction_word_count_pos_AUX_count', 'interaction_word_count_pos_ADJ_count', 'interaction_word_count_pos_VERB_count', 'interaction_word_count_pos_PUNCT_count', 'interaction_word_count_deberta_prob_4_full', 'interaction_word_count_dep_punct_count', 'interaction_word_count_deberta_prob_3_full', 'interaction_word_count_pos_ADP_count', 'interaction_word_count_dep_prep_count', 'interaction_word_count_dep_ROOT_count', 'interaction_word_count_sentence_count', 'interaction_word_count_dep_pobj_count', 'interaction_word_count_dep_nsubj_count', 'interaction_word_count_dep_aux_count', 'interaction_word_count_dep_advmod_count', 'interaction_word_count_dep_amod_count', 'interaction_word_count_spelling_errors', 'interaction_word_count_pos_ADV_count', 'interaction_word_count_dep_dobj_count', 'interaction_word_count_pos_PART_count', 'interaction_word_co

/tmp/ipykernel_104197/3155286109.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'interaction_{feature1}_{feature2}'] = df[feature1] * df[feature2]
/tmp/ipykernel_104197/3155286109.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'interaction_{feature1}_{feature2}'] = df[feature1] * df[feature2]
/tmp/ipykernel_104197/3155286109.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

In [25]:
df.head()

essay_id                                          full_text  score  \
0  000d118  Many people have car where they live. The thin...      3   
1  000fe60  I am a scientist at NASA that is discussing th...      3   
2  001ab80  People always wish they had the same technolog...      4   
3  001bdc0  We all heard about Venus, the planet without a...      4   
4  002ba53  Dear, State Senator\n\nThis is a letter to arg...      3   

                                             lowered  \
0  many people have car where they live. the thin...   
1  i am a scientist at nasa that is discussing th...   
2  people always wish they had the same technolog...   
3  we all heard about venus, the planet without a...   
4  dear, state senator\n\nthis is a letter to arg...   

                                          clean_text  paragraph_count  \
0  many people have car where they live the thing...                1   
1  i am a scientist at nasa that is discussing th...                5   
2  people always wish they had the same technolog...                4   
3  we all heard about venus the planet without al...                5   
4  dear state senator this is a letter to argue i...                6   

   sentence_count  word_count  avg_paragraph_length  max_paragraph_length  \
0              13         498            498.000000                   498   
1              21         332             66.400000                    98   
2              24         550            137.500000                   199   
3              20         451             90.200000                   165   
4              15         373             62.166667                   118   

   min_paragraph_length  avg_sentence_length  max_sentence_length  \
0                   498            38.307692                  127   
1                    37            15.809524                   48   
2                    85            22.916667                   46   
3                    25            22.550000                   38   
4                     2            24.933333                   76   

   min_sentence_length  avg_word_length  max_word_length  min_word_length  \
0                    7         4.369478               25                1   
1                    2         4.018072               11                1   
2                    9         4.574545               15                1   
3                    5         4.982262               20                1   
4                    2         4.873995               14                1   

   flesch_reading_ease  gunning_fog_index  sentiment_score  \
0                57.98              17.33           0.9937   
1                87.55               7.48           0.7705   
2                65.15              11.49          -0.9731   
3                58.32              11.91           0.9702   
4                54.66              12.64           0.9771   

   deberta_prob_0_full  deberta_prob_1_full  deberta_prob_2_full  \
0             0.021534             0.379466             0.529225   
1             0.001421             0.071248             0.839510   
2             0.001564             0.001929             0.033057   
3             0.000927             0.005752             0.206941   
4             0.031210             0.388895             0.522782   

   deberta_prob_3_full  deberta_prob_4_full  deberta_prob_5_full  \
0             0.066918             0.002071             0.000785   
1             0.086768             0.000765             0.000288   
2             0.516632             0.433214             0.013603   
3             0.767750             0.017782             0.000848   
4             0.054104             0.002046             0.000963   

   xlnet_prob_0  xlnet_prob_1  xlnet_prob_2  xlnet_prob_3  xlnet_prob_4  \
0      0.118853      0.174859      0.158128      0.286387      0.150762   
1      0.116731      0.167865      0.149173      0.298451      0.157676   
2      0.121194      0.172987      0

In [26]:
df.tail()


essay_id                                          full_text  score  \
17302  ffd378d  the story " The Challenge of Exploing Venus " ...      2   
17303  ffddf1f  Technology has changed a lot of ways that we l...      4   
17304  fff016d  If you don't like sitting around all day than ...      2   
17305  fffb49b  In "The Challenge of Exporing Venus," the auth...      1   
17306  fffed3e  Venus is worthy place to study but dangerous. ...      2   

                                                 lowered  \
17302  the story " the challenge of exploing venus " ...   
17303  technology has changed a lot of ways that we l...   
17304  if you don't like sitting around all day than ...   
17305  in "the challenge of exporing venus," the auth...   
17306  venus is worthy place to study but dangerous. ...   

                                              clean_text  paragraph_count  \
17302  the story the challenge of exploing venus is a...                3   
17303  technology has changed a lot of ways that we l...                6   
17304  if you do not like sitting around all day than...                3   
17305  in the challenge of exporing venus the author ...                1   
17306  venus is worthy place to study but dangerous t...                4   

       sentence_count  word_count  avg_paragraph_length  max_paragraph_length  \
17302               9         157             52.333333                    82   
17303              26         579             96.500000                   167   
17304              15         215             71.666667                    86   
17305              11         231            231.000000                   231   
17306              11         155             38.750000                    94   

       min_paragraph_length  avg_sentence_length  max_sentence_length  \
17302                    23            17.444444                   36   
17303                    44            22.269231                   61   
17304                    63            14.333333                   23   
17305                   231            21.000000                   38   
17306                     3            14.090909                   28   

       min_sentence_length  avg_word_length  max_word_length  min_word_length  \
17302                    7         4.445860               12                1   
17303                    1         4.770294               12                1   
17304                    5         4.213953               11                1   
17305                   11         5.181818               12                1   
17306                    6         4.051613               10                1   

       flesch_reading_ease  gunning_fog_index  sentiment_score  \
17302                70.94               9.72           0.3626   
17303                56.39              11.42           0.9973   
17304                90.80               6.28           0.9803   
17305                58.72              11.66           0.9513   
17306                83.76               5.93           0.8294   

       deberta_prob_0_full  deberta_prob_1_full  deberta_prob_2_full  \
17302             0.092310             0.891637             0.014393   
17303             0.001012             0.005224             0.197986   
17304             0.043474             0.935611             0.019348   
17305             0.675936             0.293551             0.025413   
17306             0.131172             0.857495             0.009533   

       deberta_prob_3_full  deberta_prob_4_full  deberta_prob_5_full  \
17302             0.000741             0.000411             0.000507   
17303             0.711231             0.082317             0.002229   
17304             0.000795             0.000298             0.000474   
17305             0.002706             0.001254             0.001139   
17306             0.000750             0.000413             0.000638   

       xlnet_prob_0  xlnet_prob_1  xlnet_prob_2  

In [27]:
# drop columns with hash in the name

hash_cols = [col for col in deberta_full.columns if 'hash' in col]

df.drop(columns=hash_cols, inplace=True)

import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer


# Initialize the HashingVectorizer
n_features = 200  # Number of columns in the hashed vector
hashing_vectorizer = HashingVectorizer(n_features=n_features, alternate_sign=False)

# Transform the text column into hash vector features
hashed_features = hashing_vectorizer.transform(df[text_col])

# Convert the hashed features to a dense representation for easier manipulation
hashed_features_dense = hashed_features.todense()

# Create new column names based on the number of features
hashed_columns = [f'hash_{i}' for i in range(n_features)]

# Convert dense representation to a dataframe
hashed_df = pd.DataFrame(hashed_features_dense, columns=hashed_columns)

# Concatenate the original dataframe with the hashed dataframe

df = pd.concat([df, hashed_df], axis=1)

In [28]:
train = df.copy()

In [29]:
train.head()

essay_id                                          full_text  score  \
0  000d118  Many people have car where they live. The thin...      3   
1  000fe60  I am a scientist at NASA that is discussing th...      3   
2  001ab80  People always wish they had the same technolog...      4   
3  001bdc0  We all heard about Venus, the planet without a...      4   
4  002ba53  Dear, State Senator\n\nThis is a letter to arg...      3   

                                             lowered  \
0  many people have car where they live. the thin...   
1  i am a scientist at nasa that is discussing th...   
2  people always wish they had the same technolog...   
3  we all heard about venus, the planet without a...   
4  dear, state senator\n\nthis is a letter to arg...   

                                          clean_text  paragraph_count  \
0  many people have car where they live the thing...                1   
1  i am a scientist at nasa that is discussing th...                5   
2  people always wish they had the same technolog...                4   
3  we all heard about venus the planet without al...                5   
4  dear state senator this is a letter to argue i...                6   

   sentence_count  word_count  avg_paragraph_length  max_paragraph_length  \
0              13         498            498.000000                   498   
1              21         332             66.400000                    98   
2              24         550            137.500000                   199   
3              20         451             90.200000                   165   
4              15         373             62.166667                   118   

   min_paragraph_length  avg_sentence_length  max_sentence_length  \
0                   498            38.307692                  127   
1                    37            15.809524                   48   
2                    85            22.916667                   46   
3                    25            22.550000                   38   
4                     2            24.933333                   76   

   min_sentence_length  avg_word_length  max_word_length  min_word_length  \
0                    7         4.369478               25                1   
1                    2         4.018072               11                1   
2                    9         4.574545               15                1   
3                    5         4.982262               20                1   
4                    2         4.873995               14                1   

   flesch_reading_ease  gunning_fog_index  sentiment_score  \
0                57.98              17.33           0.9937   
1                87.55               7.48           0.7705   
2                65.15              11.49          -0.9731   
3                58.32              11.91           0.9702   
4                54.66              12.64           0.9771   

   deberta_prob_0_full  deberta_prob_1_full  deberta_prob_2_full  \
0             0.021534             0.379466             0.529225   
1             0.001421             0.071248             0.839510   
2             0.001564             0.001929             0.033057   
3             0.000927             0.005752             0.206941   
4             0.031210             0.388895             0.522782   

   deberta_prob_3_full  deberta_prob_4_full  deberta_prob_5_full  \
0             0.066918             0.002071             0.000785   
1             0.086768             0.000765             0.000288   
2             0.516632             0.433214             0.013603   
3             0.767750             0.017782             0.000848   
4             0.054104             0.002046             0.000963   

   xlnet_prob_0  xlnet_prob_1  xlnet_prob_2  xlnet_prob_3  xlnet_prob_4  \
0      0.118853      0.174859      0.158128      0.286387      0.150762   
1      0.116731      0.167865      0.149173      0.298451      0.157676   
2      0.121194      0.172987      0

In [30]:
len(train.columns)

2895

In [ ]:
# import gc

# del df_train_final, df, df_ngrams, df_svd, df_pos, df_entities, df_deps, pos_df, ner_df, dep_df, pos_features, ner_features, dep_features

# gc.collect()

In [31]:
# show columns with nan values greater than 0

nan_cols = train.isna().sum()

nan_cols = nan_cols[nan_cols > 0]

nan_cols    


Series([], dtype: int64)

In [33]:
drop_cols = ['full_text', 'clean_text', 'lowered']   # processed_docs', , 'global_coherence', 'topics'


train_df = train.copy()

train_df.drop(columns=drop_cols, inplace= True)


In [34]:
train_df.head()

essay_id  score  paragraph_count  sentence_count  word_count  \
0  000d118      3                1              13         498   
1  000fe60      3                5              21         332   
2  001ab80      4                4              24         550   
3  001bdc0      4                5              20         451   
4  002ba53      3                6              15         373   

   avg_paragraph_length  max_paragraph_length  min_paragraph_length  \
0            498.000000                   498                   498   
1             66.400000                    98                    37   
2            137.500000                   199                    85   
3             90.200000                   165                    25   
4             62.166667                   118                     2   

   avg_sentence_length  max_sentence_length  min_sentence_length  \
0            38.307692                  127                    7   
1            15.809524                   48                    2   
2            22.916667                   46                    9   
3            22.550000                   38                    5   
4            24.933333                   76                    2   

   avg_word_length  max_word_length  min_word_length  flesch_reading_ease  \
0         4.369478               25                1                57.98   
1         4.018072               11                1                87.55   
2         4.574545               15                1                65.15   
3         4.982262               20                1                58.32   
4         4.873995               14                1                54.66   

   gunning_fog_index  sentiment_score  deberta_prob_0_full  \
0              17.33           0.9937             0.021534   
1               7.48           0.7705             0.001421   
2              11.49          -0.9731             0.001564   
3              11.91           0.9702             0.000927   
4              12.64           0.9771             0.031210   

   deberta_prob_1_full  deberta_prob_2_full  deberta_prob_3_full  \
0             0.379466             0.529225             0.066918   
1             0.071248             0.839510             0.086768   
2             0.001929             0.033057             0.516632   
3             0.005752             0.206941             0.767750   
4             0.388895             0.522782             0.054104   

   deberta_prob_4_full  deberta_prob_5_full  xlnet_prob_0  xlnet_prob_1  \
0             0.002071             0.000785      0.118853      0.174859   
1             0.000765             0.000288      0.116731      0.167865   
2             0.433214             0.013603      0.121194      0.172987   
3             0.017782             0.000848      0.120910      0.168503   
4             0.002046             0.000963      0.117731      0.174473   

   xlnet_prob_2  xlnet_prob_3  xlnet_prob_4  xlnet_prob_5  use_embedding_0  \
0      0.158128      0.286387      0.150762      0.111011         0.041960   
1      0.149173      0.298451      0.157676      0.110105        -0.051479   
2      0.152385      0.294505      0.148131      0.110798        -0.001355   
3      0.148743      0.297890      0.153702      0.110252         0.024647   
4      0.146372      0.301475      0.148458      0.111491        -0.052357   

   use_embedding_1  use_embedding_2  use_embedding_3  use_embedding_4  \
0        -0.067975         0.063115        -0.026617         0.020445   
1        -0.021961         0.028500         0.046144         0.066032   
2        -0.050998        -0.028374        -0.011676        -0.022839   
3        -0.046599        -0.010909         0.027460         0.060216   
4        -0.057141        -0.007919        -0.028717         0.057074   

   use_embedding_5  use_embedding_6  use_embedding_7  use_embedding_8  \
0         0.003867         0.009433        -0.055062        -0.053429   
1        -0.067002 

In [35]:
from sklearn.model_selection import train_test_split

# Features and target variable
X = train_df.drop('score', axis=1)
y = train_df['score']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [36]:
feature_cols = []

for col in X_train.columns:
    if (col != 'essay_id') and (col != 'score'):
        feature_cols.append(col) 

In [37]:
# pd.set_option('display.max_columns', None)
# feature_cols

In [38]:
# save the order of the column names to a txt file

with open(file_config['feature_cols'], 'w') as f:
    for item in feature_cols:
        f.write("%s\n" % item)

In [39]:
import sklearn
print(sklearn.__version__)

1.2.2


In [40]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pickle
import numpy as np

# Extract labels from the training and validation datasets
train_labels = y_train.copy()
val_labels = y_test.copy()

# Extract features for scaling
train_features = X_train[feature_cols] # Subset with only feature columns
val_features = X_test[feature_cols]

# Initialize the scaler
scaler = StandardScaler()  # Using StandardScaler for scaling

# Fit the scaler to the training features
scaler.fit(train_features)  # This defines the transformation based on the training data

# Transform training and validation features
train_feats_scaled = scaler.transform(train_features)  # Transforms the training data
val_feats_scaled = scaler.transform(val_features)  # Transforms the validation data

# Reassign the scaled features to the original DataFrames, keeping the same column names
X_train[feature_cols] = train_feats_scaled  # Replace the original features with scaled ones
X_test[feature_cols] = val_feats_scaled

# Save the scaler for later use
with open(file_config['scaler'], 'wb') as f:
    pickle.dump(scaler, f)  # Persist the scaler for future use or reference


In [41]:
print(X_train.shape, X_test.shape)

(12114, 2891) (5193, 2891)


In [42]:
# Check if the file has been written correctly and is not empty
import os

scaler_path = file_config['scaler']

if os.path.getsize(scaler_path) > 0:

    print(f"Scaler saved successfully in {scaler_path}.")
    
else:

    print(f"Failed to save scaler to {scaler_path}. File is empty.")

Scaler saved successfully in /home/jack/github/kaggle/scoring/model_data/sklearn/scaler.pkl.


In [43]:
# Check if the scaler is StandardScaler
if isinstance(scaler, StandardScaler):
    print("The scaler is a StandardScaler.")
elif isinstance(scaler, MinMaxScaler):
    print("The scaler is a MinMaxScaler.")
else:
    print("The scaler is neither StandardScaler nor MinMaxScaler.")

The scaler is a StandardScaler.


In [44]:
# To convert features and targets to NumPy arrays for ML use

# train_features = X_train.values

# train_labels = np.array(train_labels.values)

# val_features = X_test.values

# val_labels = np.array(y_test.values)




# print("Features shape:", train_features.shape)
# print("Target shape:", train_labels.shape)

# print("Features shape:", val_features.shape)
# print("Target shape:", val_labels.shape)



#####     Features shape: (3, 1389)


In [45]:
import pandas as pd
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTE
from sklearn import ensemble, model_selection, metrics
import numpy as np
import pickle
import os
from sklearn.base import clone
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import make_scorer, accuracy_score, cohen_kappa_score
import keras_tuner

# Function to compute Quadratic Weighted Kappa
def quadratic_weighted_kappa_scorer(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

# Define the model building function for the tuner
def build_model(hp):
    model_type = hp.Choice('model_type', ['random_forest', 'gradient_boosting', 'lightgbm'])
    if model_type == 'random_forest':
        model = ensemble.RandomForestClassifier(
            n_estimators=hp.Int('n_estimators', 10, 100, step=10),
            max_depth=hp.Int('max_depth', 3, 20),
            min_samples_split=hp.Int('min_samples_split', 2, 20),
            min_samples_leaf=hp.Int('min_samples_leaf', 1, 10),
            criterion=hp.Choice('criterion', ['gini', 'entropy']),
            class_weight=hp.Choice('class_weight', ['balanced', 'balanced_subsample']),
            max_samples=hp.Float('max_samples', 0.1, 1.0, sampling='log'))
    elif model_type == 'gradient_boosting':
        model = ensemble.GradientBoostingClassifier(
            n_estimators=hp.Int('n_estimators', 50, 200, step=50),
            learning_rate=hp.Float('learning_rate', 0.01, 0.2, sampling='log'),
            max_depth=hp.Int('max_depth', 3, 15),
            subsample=hp.Float('subsample', 0.5, 1.0, step=0.1))
    else:  # LightGBM
        from lightgbm import LGBMClassifier
        model = LGBMClassifier(
            n_estimators=hp.Int('n_estimators', 50, 200, step=50),
            num_leaves=hp.Int('num_leaves', 31, 127, step=16),
            learning_rate=hp.Float('learning_rate', 0.01, 0.2, sampling='log'),
            min_child_samples=hp.Int('min_child_samples', 10, 50, step=10),
            max_depth=hp.Int('max_depth', 3, 15),  # Tunable max_depth parameter
            min_data_in_leaf=hp.Int('min_data_in_leaf', 20, 100, step=10),  # Increases the minimum data per leaf
            min_gain_to_split=hp.Float('min_gain_to_split', 0.001, 0.1, sampling='log'),  # Ensures splits bring gains
            lambda_l1=hp.Float('lambda_l1', 0.01, 10.0, sampling='log'),  # L1 regularization
            lambda_l2=hp.Float('lambda_l2', 0.01, 10.0, sampling='log'),  # L2 regularization
            force_col_wise=True,  # Force using column-wise calculation to avoid overhead
            class_weight='balanced'  # Handling class imbalance
        )
    return model





# Prepare data: Assuming X_train and y_train are already defined
X_train_original = X_train.drop(columns='essay_id').copy()  # Keep the original training data
y_train_original = y_train.drop(columns='essay_id').copy() 

# Apply SMOTE to balance the minority classes
smote = SMOTE(sampling_strategy='not majority', random_state=42)
X_train_upsampled, y_train_upsampled = smote.fit_resample(X_train_original, y_train_original)


2024-05-27 17:46:38.826113: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-27 17:46:39.674064: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [46]:
# Tuner configuration
tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=10),
    hypermodel=build_model,
    scoring=make_scorer(quadratic_weighted_kappa_scorer),
    cv=model_selection.StratifiedKFold(5),
    directory='.',
    project_name='data/sklearn',
    overwrite=True)


# Tuning on the original data
tuner.search(X_train_original.values, y_train_original.values)



Trial 4 Complete [00h 00m 24s]
score: 0.8972618410327252

Best score So Far: 0.902710733539065
Total elapsed time: 00h 03m 39s

Search: Running Trial #5

Value             |Best Value So Far |Hyperparameter
gradient_boosting |lightgbm          |model_type
20                |60                |n_estimators
10                |18                |max_depth
5                 |3                 |min_samples_split
10                |7                 |min_samples_leaf
gini              |entropy           |criterion
balanced_subsample|balanced_subsample|class_weight
0.50961           |0.12221           |max_samples
111               |127               |num_leaves
0.12777           |0.021199          |learning_rate
50                |50                |min_child_samples
60                |80                |min_data_in_leaf
0.038145          |0.0016527         |min_gain_to_split
0.17072           |0.44372           |lambda_l1
0.59275           |0.010688          |lambda_l2



In [ ]:
# # apply smote within k-fold cross validation in order to avoid data leakage

# # Define the cross-validation strategy

# cv = model_selection.StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # Initialize the model
# model = ensemble.RandomForestClassifier(random_state=42)

# # Initialize the SMOTE object

# smote = SMOTE(sampling_strategy='not majority', random_state=42)

# # Initialize the StandardScaler
# scaler = StandardScaler()

# # Initialize the list to store the cross-validated scores

# cv_scores = []

# # Perform cross-validation

# for train_idx, val_idx in cv.split(X_train, y_train):

#     # Extract the training and validation data
#     X_train_fold, y_train_fold = X_train.iloc[train_idx], y_train.iloc[train_idx]
#     X_val_fold, y_val_fold = X_train.iloc[val_idx], y_train.iloc[val_idx]

#     # Apply SMOTE to the training data
#     X_train_resampled, y_train_resampled = smote.fit_resample(X_train_fold, y_train_fold)

#     # Scale the training and validation data
#     X_train_scaled = scaler.fit_transform(X_train_resampled)
#     X_val_scaled = scaler.transform(X_val_fold)

#     # Fit the model on the training data
#     model.fit(X_train_scaled, y_train_resampled)

#     # Predict the target on the validation data
#     y_pred = model.predict(X_val_scaled)

#     # Compute the Quadratic Weighted Kappa
#     qwk = quadratic_weighted_kappa_scorer(y_val_fold, y_pred)

#     # Append the score to the list
#     cv_scores.append(qwk)

# # Compute the average score

# cv_score = np.mean(cv_scores)

# # Print the average score

# print(f"Average Quadratic Weighted Kappa: {cv_score:.4f}")

# # Save the model

# with open(file_config['sklearn_model'], 'wb') as f:
#     pickle.dump(model, f)



In [ ]:
# Retrieve the best model
best_model = tuner.get_best_models(num_models=1)[0]

# Fit models on both upsampled and original data
model_standard = clone(best_model)
model_standard.fit(X_train_original, y_train_original)

model_upsampled = clone(best_model)
model_upsampled.fit(X_train_upsampled, y_train_upsampled)

# Ensemble using voting
ensemble_model = VotingClassifier(estimators=[
    ('standard', model_standard),
    ('upsampled', model_upsampled)
], voting='soft')



In [ ]:
# Assuming X_val and y_val are your validation sets

ensemble_model.fit(X_train_original, y_train_original)  # Optionally refit if necessary

In [ ]:
predictions = ensemble_model.predict(X_test)

# Evaluate and print results
print("Validation Accuracy:", accuracy_score(val_labels, predictions))
print("Validation Cohen's Kappa:", quadratic_weighted_kappa_scorer(val_labels, predictions))
print("Validation Quadratic Weighted Kappa:", quadratic_weighted_kappa_scorer(val_labels, predictions))
print("Validation Cohens Weighted Kappa:",cohen_kappa_score(val_labels, predictions, weights='quadratic'))

In [ ]:
# confusion matrix

from sklearn.metrics import confusion_matrix

confusion_matrix(val_labels, predictions)

In [ ]:
# classification report

from sklearn.metrics import classification_report

print(classification_report(val_labels, predictions))

In [ ]:
from joblib import dump, load

dump(ensemble_model, file_config['sklearn_model'])


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Generate and display the confusion matrix for the test predictions
cm = confusion_matrix(val_labels, predictions, labels=best_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()


In [ ]:
X_test['predictions'] = predictions

df_aggregated = X_test.groupby('essay_id')['predictions'].agg(lambda x: x.value_counts().index[0]).reset_index()

df_aggregated.head()

In [ ]:
# Identifying misclassified samples
misclassified_indices = (y_true != predictions)
misclassified_samples = val_features[misclassified_indices]
misclassified_true_labels = y_true[misclassified_indices]
misclassified_predicted_labels = predictions[misclassified_indices]


In [ ]:
# Examining features of misclassified samples
import pandas as pd
misclassified_df = pd.DataFrame(misclassified_samples)
misclassified_df['True_Label'] = misclassified_true_labels
misclassified_df['Predicted_Label'] = misclassified_predicted_labels
# Display the first few rows of the misclassified samples
misclassified_df.head()

In [ ]:
# # Feature importance from LightGBM
# feature_importance = best_model.feature_importance(importance_type='gain')
# feature_names = val_features.columns
# feature_importance_df = pd.DataFrame({
#     'Feature': feature_names,
#     'Importance': feature_importance
# })
# # Sort features by importance
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
# print(feature_importance_df)


In [ ]:
# Checking class distribution in the true labels
label_counts = pd.Series(y_true).value_counts()
print(label_counts)


In [ ]:
# from sklearn.model_selection import GridSearchCV
# # Define the hyperparameters to tune
# param_grid = {
#     'learning_rate': [0.01, 0.05, 0.1],
#     'num_leaves': [31, 62, 128],
#     'n_estimators': [50, 100, 200]
# }
# grid_search = GridSearchCV(best_model, param_grid, scoring='', cv=5)
# grid_search.fit(val_features, val_labels)
# print("Best parameters:", grid_search.best_params_)


In [ ]:
# Check LightGBM version
import lightgbm as lgb
print(lgb.__version__)


In [ ]:
# # Ensure the model is fitted
# if not hasattr(best_model, 'feature_importance_'):
#     best_model.fit(val_features, val_labels)  # Train or re-train the model

# # Retrieve feature importance
# feature_importance = best_model.feature_importance_  # Underscore at the end
# feature_names = val_features.columns

# # Create DataFrame for feature importance
# feature_importance_df = pd.DataFrame({
#     'Feature': feature_names,
#     'Importance': feature_importance
# })
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# print(feature_importance_df)


In [ ]:
# Assuming `df` is the original DataFrame and `predictions` is a list/array of predicted labels
# If there's a subset used for predictions, ensure it aligns with the DataFrame
val_df['Predicted_Label'] = predictions


In [ ]:
print(len(val_features), len(val_df), len(predictions))

In [ ]:
val_df.tail()

In [ ]:
# merge 'clean_text' column back with val df 

text = val_essays[['essay_id', 'clean_text']]

val_df = val_df.merge(text, on = 'essay_id')

val_df.head()

In [ ]:
# Assuming `df` has the original labels in a column named 'True_Label'
misclassified_samples = val_df[val_df['score'] != val_df['Predicted_Label']]

# Display misclassified samples
print(misclassified_samples[['score', 'Predicted_Label', 'clean_text']][:15])


In [ ]:
# Example: Analyze original text of misclassified samples
misclassified_text = misclassified_samples['clean_text']  # Assuming 'Text' is the original text data
print(misclassified_text.head())


In [ ]:
# load tf model
len(misclassified_text)

In [ ]:
# add full mis classified text to a txt file seperated by  '---' and a new line after every 10 words in each essay

with open('misclassified_text.txt', 'w') as f:
    for essay in misclassified_text:
        f.write("%s\n" % essay)
        f.write('---\n')
        


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Assuming `df` has your model's features and labels
X = val_df[feature_cols]
y = val_df['score'] == val_df['Predicted_Label']

# Splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit a RandomForest to see feature importances
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Get the most important features
importances = clf.feature_importances_
features = feature_cols
important_features = {features[i]: importances[i] for i in range(len(features))}

# convert to percentages rounded and sort

important_features = {k: round(v, 5) for k, v in important_features.items()}

print("Most predictive features of misclassification:", sorted(important_features.items(), key=lambda x: x[1], reverse=True))

In [ ]:
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

# # Assume val_df and feature_cols are already defined

# # Set the option to display all rows
# pd.set_option('display.max_rows', None)

# # Selecting the features and the target
# X = val_df[feature_cols]
# y = val_df['score']

# # Calculate the correlation between features and the target
# corr = X.corrwith(y)

# # Sort the correlation values
# corr = corr.sort_values(ascending=False)

# # Convert the Series to a DataFrame for heatmap compatibility
# corr_df = pd.DataFrame(corr, columns=['Correlation'])

# corr_df


In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report

# # Define the feature set with and without the specific features
# X_with = val_df[feature_cols]  # including all features
# X_without = val_df.drop(columns=['misspelling_count', 'sentence_length'], axis=1) # excluding problematic features

# y = val_df['score']  # target variable

# # Split the data
# X_train_with, X_test_with, y_train, y_test = train_test_split(X_with, y, test_size=0.3, random_state=42)
# X_train_without, X_test_without, _ , _ = train_test_split(X_without, y, test_size=0.3, random_state=42)

# # Train models
# model_with = RandomForestClassifier().fit(X_train_with, y_train)
# model_without = RandomForestClassifier().fit(X_train_without, y_train)

# # Evaluate models
# print("With features:")
# print(classification_report(y_test, model_with.predict(X_test_with)))

# print("Without features:")
# print(classification_report(y_test, model_without.predict(X_test_without)))
